# Downloading IBDMDB Data

This notebook demonstrates how to download raw sequencing data from the Inflammatory Bowel Disease Multi-omics Database (https://ibdmdb.org/) using basic web scraping techniques in Python.

## Goals

1. Become familiar with sending HTTP requests using 'requests'.
2. Learn how to extract data from HTML using 'BeautifulSoup'.

# 1. Load packages

In [2]:
import requests #The requests module allows you to send HTTP requests using Python
from bs4 import BeautifulSoup #Beautiful Soup is a Python library for pulling data out of HTML and XML files. It provides tools for parsing and navigating the parse tree, making it easy to extract data from web pages.
from urllib.parse import urljoin, urlparse #The urllib.parse module provides functions for manipulating URLs, including joining base URLs with relative URLs to create absolute URLs.
import os #The os module provides a way of using operating system dependent functionality like reading or writing to the file system, handling file paths, and managing directories.
from pathlib import Path #The pathlib allows you to work with file paths in an object-oriented way, making it easier to manipulate and interact with file system paths across different operating systems.

In [ ]:
# Paths setup

# Base path = where the project lives (works in local + Colab if consistent)
base_path = Path.cwd().resolve().parent

print("Current working directory:", base_path)

# Output directory
output_dir = base_path / "dataset" / "ibdmdb_hmp2" / "raw_sequences"

# Create folder if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

# Change working directory
os.chdir(output_dir)

print("Now working in:", os.getcwd())

Current working directory: /Users/hector/git/predictive_microbiology_workshop_UNAM
Now working in: /Users/hector/git/predictive_microbiology_workshop_UNAM/dataset/ibdmdb_hmp2/hmp2_raw_16s_dataset


In [2]:
# Download raw files (sequences) from IBDMDB

url= "https://ibdmdb.org/downloads/html/rawfiles_16s_2018-01-08.html"

response = requests.get(url)
print('Status code:', response.status_code) # if 200, the request was successful

Status code: 200


In [3]:
# Parse the HTML content of the page to find the download links for the raw files

soup = BeautifulSoup(response.text, "html.parser")
print(soup) # print the HTML content of the page to understand its structure and identify the relevant elements containing the download links.

<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Results</title>
<link href="/static/css/bootstrap-theme.min.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/bootstrap.min.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/ibdmdb.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/mezzanine.css" rel="stylesheet" type="text/css"/>
</head>
<body>
<div class="navbar navbar-inverse navbar-fixed-top" role="navigation">
<div class="container-fluid">
<div class="navbar-header">
<a class="navbar-brand" href="/" style="padding: 0px 10px;">
<img alt="IBDMDB.org" height="50" src="/static/img/gut_cartoon.png" width="50"/>
</a>
<ul class="nav navbar-nav">
<li id="dropdown-menu-home">
<a href="/">Home</a>
</li>
<li id="dropdown-menu-home">
<a href="/results">Download Data</a>
</li>
<li id="dropdown-menu-home">
<a href="/protocols">Protocols</a>
</li>
<li id="dropdown-menu-home">
<a href="/project">Team</a>
</li>
<li id="dropdown-menu-home">
<a hre

In [4]:

links = []
# iterate over the elements of the webpage 'a' finds all anchor tags with an href attribute, which typically represent hyperlinks in HTML. The href attribute contains the URL of the linked resource.
for a in soup.find_all('a', href=True): 
    link = a['href'] # extract the URL of the linked resource
    # check if the link ends with '.fastq.gz', which is a common file extension for compressed FASTQ files (sequences), often used for storing biological sequence data.
    if link.endswith('.fastq.gz'):
        full_link = urljoin(url, link) # Convert relative URL to absolute URL
        links.append(full_link)
        print('Download link found:', full_link)

Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206534.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206536.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206538.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206547.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206548.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206561.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206562.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206563.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206564.f

In [ ]:
# print the current working directory to confirm where the files will be saved

print(os.getcwd()) 

# Make output directory for downloaded files
os.makedirs("qz_files", exist_ok=True)

# Change the current working directory to the desired output directory
os.chdir("/Users/hector/git/predictive_microbiology_workshop_UNAM/dataset/ibdmdb_hmp2_raw_16s_dataset_v1/qz_files")

/Users/hector/Documents/IE_UNAM/Taller_microbiologia_predictiva


In [ ]:
# Download each file from the list of links

n= len(links) # get the total number of download links found on the webpage
#n = 10  # number of files to download

for i, url in enumerate(links[:n]): # iterate over the first n links in the list of download links
    response = requests.get(url, stream=True) # send a GET request to the URL to download the file. The stream=True parameter allows you to download the file in chunks, which is useful for large files.
    response.raise_for_status() # check if the request was successful (status code 200). If the request was not successful, this will raise an HTTPError exception.

    filename = os.path.basename(urlparse(url).path) # extract the filename from the URL using urlparse to parse the URL and os.path.basename to get the base name of the file from the path component of the URL.
    filepath = os.path.join(os.getcwd(), filename) # create the full file path by joining the current working directory with the filename.

    print(f"Downloading file {i}: {filename}") # print a message indicating which file is being downloaded, including the index of the file in the list and the filename.

    with open(filepath, "wb") as f: # open the file in binary write mode to save the downloaded content. The with statement ensures that the file is properly closed after writing.
        for chunk in response.iter_content(chunk_size=8192): # iterate over the content of the response in chunks of 8192 bytes (8 KB) using the iter_content method. This allows you to download large files without consuming too much memory.
            f.write(chunk) # write each chunk of data to the file until the entire file is downloaded.

    print("File downloaded successfully:", filename)
print("All files downloaded successfully.")

File downloaded successfully: 206534.fastq.gz
File downloaded successfully: 206536.fastq.gz
File downloaded successfully: 206538.fastq.gz
File downloaded successfully: 206547.fastq.gz
File downloaded successfully: 206548.fastq.gz
File downloaded successfully: 206561.fastq.gz
File downloaded successfully: 206562.fastq.gz
File downloaded successfully: 206563.fastq.gz
File downloaded successfully: 206564.fastq.gz
File downloaded successfully: 206569.fastq.gz
File downloaded successfully: 206570.fastq.gz
File downloaded successfully: 206571.fastq.gz
File downloaded successfully: 206572.fastq.gz
File downloaded successfully: 206603.fastq.gz
File downloaded successfully: 206604.fastq.gz
File downloaded successfully: 206605.fastq.gz
File downloaded successfully: 206608.fastq.gz
File downloaded successfully: 206609.fastq.gz
File downloaded successfully: 206614.fastq.gz
File downloaded successfully: 206615.fastq.gz
File downloaded successfully: 206616.fastq.gz
File downloaded successfully: 2066